In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 7, 6, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 7, 6, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 202.62it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name,file_date
0,4027853074000000201,,NEWT,TRAD,2026-07-06 04:00:15+00:00,None,IR,None,I,False,...,1.0,,,NaN,None,None,QZB883814F13,NA/Swap OIS INR,INR-MIBOR-OIS Compound,2026-07-06
1,4027853075000000301,,NEWT,TRAD,2026-07-06 04:00:15+00:00,None,IR,None,I,False,...,1.0,,,NaN,None,None,QZB883814F13,NA/Swap OIS INR,INR-MIBOR-OIS Compound,2026-07-06
2,4027853505000000101,,NEWT,TRAD,2026-07-06 04:01:06+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZPB1RXQ05RR,NA/Swap OIS INR,INR-MIBOR-OIS-COMPOUND,2026-07-06
3,4027854524000000201,,NEWT,TRAD,2026-07-06 04:01:07+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZXX1LPTTL0T,NA/Swap OIS INR,INR-FBIL-MIBOR-OIS-COMPOUND,2026-07-06
4,4027853838000000201,,NEWT,TRAD,2026-07-06 04:01:56+00:00,None,IR,None,I,True,...,NaN,0.7575,,3.0,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,2026-07-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15364,4046234205000000101,,NEWT,TRAD,2026-07-07 03:52:58+00:00,None,IR,None,I,True,...,1.0,None,None,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,NaN
15365,4046234206000000201,,NEWT,TRAD,2026-07-07 03:52:58+00:00,None,IR,None,I,True,...,1.0,None,None,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,NaN
15366,4046234209000000501,,NEWT,TRAD,2026-07-07 03:52:58+00:00,None,IR,None,I,True,...,1.0,None,None,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,NaN
15367,4046235559000000201,,NEWT,TRAD,2026-07-07 03:53:26+00:00,None,IR,None,I,True,...,1.0,None,None,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,NaN


In [8]:
[col for col in list(df.columns) if "fixed" in str(col).lower()]

['Fixed rate-Leg 1',
 'Fixed rate-Leg 2',
 'Fixed rate day count convention-leg 1',
 'Fixed rate day count convention-leg 2',
 'Fixed rate payment frequency period-Leg 1',
 'Fixed rate payment frequency period-Leg 2',
 'Fixed rate payment frequency period multiplier-Leg 1',
 'Fixed rate payment frequency period multiplier-Leg 2']

In [1]:
import requests
res = requests.get("https://www.astorridge.com/trade-radar-rv-trades-in-europe-26th-nov-james-rice-astor-ridge/")
res.text

'<!DOCTYPE html>\n<html class="no-touch" lang="en-GB" xmlns="http://www.w3.org/1999/xhtml">\n<head>\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<link rel="profile" href="http://gmpg.org/xfn/11">\n<link rel="pingback" href="https://www.astorridge.com/xmlrpc.php">\n<meta name=\'robots\' content=\'index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1\' />\n\n\t<!-- This site is optimized with the Yoast SEO plugin v20.9 - https://yoast.com/wordpress/plugins/seo/ -->\n\t<title>Trade Radar - RV trades in Europe 26th Nov, James Rice @Astor Ridge - Astorridge</title>\n\t<link rel="canonical" href="https://www.astorridge.com/trade-radar-rv-trades-in-europe-26th-nov-james-rice-astor-ridge/" />\n\t<meta property="og:locale" content="en_GB" />\n\t<meta property="og:type" content="article" />\n\t<meta property="og:title" content="Trade Radar - RV trades in Europe 26th Nov, James Ri

In [21]:
# df["Event timestamp"] = df["Event timestamp"].astype(str)
# df["Execution Timestamp"] = df["Execution Timestamp"].astype(str)
# df.to_csv(r"C:\Users\chris\clee\ARBS\notebooks\sdr\april_fomc_dated_sdr_trades.csv", index=False)	

In [3]:
# df[(df["Effective Date"].dt.date == datetime.date(2026, 4, 28)) & (df["Expiration Date"].dt.date == datetime.date(2026, 6, 16))]

In [9]:
# df[(df["UPI Underlier Name"].str.contains("vs", case=False, na=False))]["UPI Underlier Name"].value_counts()

# df[df["UPI Underlier Name"] == "USD-Federal Funds-OIS Compound vs USD-SOFR-OIS Compound"]

df[df["Dissemination Identifier"] == "4042593767000000101"][
    [
        "Effective Date",
        "Expiration Date",
        "Cleared",
        "Event timestamp",
        "Execution Timestamp",
        "Package transaction price",
		'Package transaction spread',
		'Exchange rate basis',
		'Spread-Leg 1',
		'Spread-Leg 2',
        'Fixed rate-Leg 1',
        'Fixed rate-Leg 2',
		# 'Price',


    ]
]

,Effective Date,Expiration Date,Cleared,Event timestamp,Execution Timestamp,Package transaction price,Package transaction spread,Exchange rate basis,Spread-Leg 1,Spread-Leg 2,Fixed rate-Leg 1,Fixed rate-Leg 2
12591,2024-03-28,2029-03-28,I,2026-07-06 19:09:19+00:00,2026-07-06 19:09:19+00:00,0,,None,0.0,-0.000255,NaN,NaN


In [1]:
list(df.columns)

NameError: name 'df' is not defined

In [13]:
basis_ois_upis = pd.read_csv(r'C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Swap-Basis_OIS.csv')
basis_upis = pd.read_csv(r'C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Swap-Basis.csv')
basis_upis = pd.concat([basis_upis, basis_ois_upis], ignore_index=True)
basis_upis

,TemplateVersion,Header_AssetClass,Header_InstrumentType,Header_UseCase,Header_Level,Identifier_UPI,Identifier_Status,Identifier_StatusReason,Identifier_LastUpdateDateTime,Derived_ClassificationType,...,Attributes_UnderlyingInstrumentIndexTermUnit,Attributes_OptionType,Attributes_UnderlyingInstrumentUPI,Attributes_OptionExerciseStyle,Attributes_ValuationMethodorTrigger,Derived_ReturnorPayoutTrigger,Attributes_UnderlyingAssetType,Attributes_UnderlyingInstrumentISIN,Attributes_UnderlierCharacteristic,Attributes_ReturnorPayoutTrigger
0,1,Rates,Swap,Basis,UPI,QZ6HLS5Z9BWB,New,NaN,2023-10-15 12:01:46,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Rates,Swap,Basis,UPI,QZZQ9QG7WFKX,New,NaN,2023-10-15 12:01:47,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Rates,Swap,Basis,UPI,QZRRB068536C,New,NaN,2023-10-15 12:02:42,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Rates,Swap,Basis,UPI,QZ18K4WXLJFN,New,NaN,2023-10-15 12:05:18,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Rates,Swap,Basis,UPI,QZTSTSHDLF6J,New,NaN,2023-10-15 12:10:05,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11100,1,Rates,Swap,Basis_OIS,UPI,QZ8F34ZHQXRK,New,NaN,2025-12-30 05:14:54,SRHCSC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11101,1,Rates,Swap,Basis_OIS,UPI,QZ4Q4WRG1691,New,NaN,2026-01-01 01:44:25,SRHCSC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11102,1,Rates,Swap,Basis_OIS,UPI,QZBSC9X5GW5C,New,NaN,2026-01-01 05:06:10,SRHDSC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11103,1,Rates,Swap,Basis_OIS,UPI,QZ6WFCMC0H9G,New,NaN,2026-01-06 08:14:51,SRHCSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df[(df["Unique Product Identifier"].isin(basis_upis["Identifier_UPI"])) & (df["UPI Underlier Name"].str.contains("USD"))][["UPI Underlier Name", "Effective Date", "Expiration Date", "Cleared", "Event timestamp", "Execution Timestamp"]]

,UPI Underlier Name,Effective Date,Expiration Date,Cleared,Event timestamp,Execution Timestamp
3455,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 08:34:33+00:00,2026-05-21 08:34:33+00:00
3500,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 08:37:00+00:00,2026-05-21 08:34:33+00:00
5017,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2036-05-26,I,2026-05-21 09:31:20+00:00,2026-05-21 09:31:20+00:00
5035,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2046-05-26,I,2026-05-21 09:32:07+00:00,2026-05-21 09:32:07+00:00
6638,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 11:14:19+00:00,2026-05-21 11:14:19+00:00
6667,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 11:15:47+00:00,2026-05-21 11:14:19+00:00
6737,USD-SOFR-OIS Compound vs USD-SOFR-OIS Compound,2026-05-28,2026-07-30,N,2026-05-21 11:20:00+00:00,2026-05-21 11:20:00+00:00
7110,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2027-05-26,2028-05-26,I,2026-05-21 11:41:53+00:00,2026-05-21 11:41:53+00:00
7169,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 11:45:09+00:00,2026-05-21 11:45:09+00:00
7196,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 11:45:55+00:00,2026-05-21 11:45:09+00:00


In [5]:
(3.812 - 3.882) - (3.843 - 3.812)

-0.10100000000000042